In [ ]:
import json
import requests
import pandas as pd
import time
from datetime import datetime
from pathlib import Path

BASE_URL = "https://play.limitlesstcg.com/api/tournaments"
TOP_LIMIT = 2
TIMEOUT = 100
EXPORT_DIR = Path("exports")


def is_rate_limited(exc):
    """True when the exception carries a 429 Too Many Requests response."""
    response = getattr(exc, "response", None)
    return response is not None and response.status_code == 429


def _flatten_nested(dataframe):
    """Serialize nested values (lists/dicts) so the Excel writer accepts them."""
    out = dataframe.copy()
    nested = (list, dict, set, tuple)
    for col in out.columns:
        if out[col].map(lambda v: isinstance(v, nested)).any():
            out[col] = out[col].map(
                lambda v: json.dumps(v, default=str) if isinstance(v, nested) else v
            )
    return out


def export_df(dataframe, name, export_dir=EXPORT_DIR):
    """Write the DataFrame to <name>_<YYYY-MM-DD>_<HHMMSS>.csv and .xlsx."""
    export_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
    csv_path = export_dir / f"{name}_{stamp}.csv"
    xlsx_path = export_dir / f"{name}_{stamp}.xlsx"

    flat = _flatten_nested(dataframe)
    flat.to_csv(csv_path, index=False)
    flat.to_excel(xlsx_path, index=False)

    print(f"Exported {name} ({len(dataframe)} rows) -> {csv_path} | {xlsx_path}")
    return csv_path, xlsx_path

all_tournaments = []
rate_limited = False

for page in range(1, 2):
    print(f"Fetching page {page}")

    r = requests.get(
        BASE_URL,
        params={
            "page": page,
            "game": "PTCG",
            "format": "STANDARD",
        },
        timeout=TIMEOUT,
    )

    try:
        r.raise_for_status()
    except requests.exceptions.HTTPError as e:
        if is_rate_limited(e):
            print(f"429 Client Error on page {page} - stopping iteration")
            rate_limited = True
            break
        raise

    tournaments = r.json()

    if not tournaments:
        break

    all_tournaments.extend(tournaments)

    time.sleep(TIMEOUT/1000)

df = pd.DataFrame(all_tournaments)

Fetching page 1


In [ ]:
df = df.drop(columns=['game'])
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
export_df(df, "tournaments")
df.head()

Exported df_clean (50 rows) -> exports\df_clean_2026-07-24_182822.csv | exports\df_clean_2026-07-24_182822.xlsx


,name,date,format,id,players,organizerId
0,Nightshades Premier Circuit Challenge #16 25 C...,2026-07-24,STANDARD,69d519dd0e548b5c2fbe594c,32,2579
1,Turtwig Den Revival Series Challenge 41,2026-07-24,STANDARD,69725552b1294bfab720364d,6,196
2,Team Corna Weekly July #5,2026-07-24,STANDARD,6a5baf52937230b102d3eb53,111,2583
3,Casual Friday!,2026-07-24,STANDARD,6a621b37937230b102d423ab,22,1216
4,Celestials Friday Casuals 07/24,2026-07-24,STANDARD,6a635f49937230b102d42f70,26,1073


In [3]:
tournament_details = []
details_rate_limited = False

# Iterating through the IDs in the existing df
for tournament_id in df['id']:
    print(f"Fetching details for: {tournament_id}")
    detail_url = f"https://play.limitlesstcg.com/api/tournaments/{tournament_id}/details"

    try:
        response = requests.get(detail_url, timeout=TIMEOUT)
        response.raise_for_status()
        tournament_details.append(response.json())
    except Exception as e:
        if is_rate_limited(e):
            print(f"429 Client Error on {tournament_id} - stopping iteration")
            details_rate_limited = True
            break
        print(f"Failed to fetch {tournament_id}: {e}")

    # Small sleep to respect the API rate limits
    time.sleep(TIMEOUT/1000)

# Create the new details DataFrame
details_df = pd.DataFrame(tournament_details)

Fetching details for: 69d519dd0e548b5c2fbe594c
Fetching details for: 69725552b1294bfab720364d
Fetching details for: 6a5baf52937230b102d3eb53
Fetching details for: 6a621b37937230b102d423ab
Fetching details for: 6a635f49937230b102d42f70
Fetching details for: 6a5925f1cc68177ec6c56c99
Fetching details for: 6a5f729b52c24ac2da6480ec
Fetching details for: 6a62967c52c24ac2da64a0c5
Fetching details for: 6a2acf672d97f3b0c2613d1c
Fetching details for: 6a60e6e3937230b102d41821
Fetching details for: 6a62800b937230b102d428c0
Fetching details for: 6a5e4d3752c24ac2da647443
Fetching details for: 6a60c513937230b102d4178c
Fetching details for: 6a50031c65724db9ded4178e
Fetching details for: 6a5e2d63937230b102d401a2
Fetching details for: 6a5e38dc937230b102d401fa
Fetching details for: 6a5a671b52c24ac2da644850
Fetching details for: 6a619256937230b102d4216b
Fetching details for: 6a613ff2937230b102d41ded
Fetching details for: 6a58f606cc68177ec6c56a6c
Fetching details for: 6a5e86f652c24ac2da6478eb
Fetching deta

In [4]:
details_df = details_df.drop(columns=['game'])
details_df['date'] = pd.to_datetime(details_df['date'], utc=True).dt.strftime('%Y-%m-%d')
details_df.head()

,id,format,name,date,players,organizer,platform,decklists,isPublic,isOnline,phases
0,69d519dd0e548b5c2fbe594c,STANDARD,Nightshades Premier Circuit Challenge #16 25 C...,2026-07-24,32,"{'id': 2579, 'name': 'Nightshades Premier Circ...",PTCGL,True,True,True,"[{'phase': 1, 'type': 'SWISS', 'rounds': 5, 'm..."
1,69725552b1294bfab720364d,STANDARD,Turtwig Den Revival Series Challenge 41,2026-07-24,6,"{'id': 196, 'name': 'DabbingUkranian'}",PTCGL,True,True,True,"[{'phase': 1, 'type': 'SWISS', 'rounds': 4, 'm..."
2,6a5baf52937230b102d3eb53,STANDARD,Team Corna Weekly July #5,2026-07-24,111,"{'id': 2583, 'name': 'Team Corna'}",PTCGL,True,True,True,"[{'phase': 1, 'type': 'SWISS', 'rounds': 7, 'm..."
3,6a621b37937230b102d423ab,STANDARD,Casual Friday!,2026-07-24,22,"{'id': 1216, 'name': 'Shiny Eevee Tourneys'}",PTCGL,True,True,True,"[{'phase': 1, 'type': 'SWISS', 'rounds': 5, 'm..."
4,6a635f49937230b102d42f70,STANDARD,Celestials Friday Casuals 07/24,2026-07-24,26,"{'id': 1073, 'name': 'Team Celestials'}",None,True,True,False,"[{'phase': 1, 'type': 'SWISS', 'rounds': 4, 'm..."


In [ ]:
# One row per tournament phase, with the phase fields as columns
phases_df = details_df.explode('phases', ignore_index=True)

phase_cols = pd.json_normalize(
    phases_df['phases'].map(lambda v: v if isinstance(v, dict) else {})
).reindex(columns=['phase', 'type', 'rounds', 'mode'])

phases_df = phases_df.drop(columns=['phases']).join(phase_cols)

export_df(phases_df, "phases")
phases_df.head()

Exported details_phases (71 rows) -> exports\details_phases_2026-07-24_182908.csv | exports\details_phases_2026-07-24_182908.xlsx


,id,format,name,date,players,organizer,platform,decklists,isPublic,isOnline,phase,type,rounds,mode
0,69d519dd0e548b5c2fbe594c,STANDARD,Nightshades Premier Circuit Challenge #16 25 C...,2026-07-24,32,"{'id': 2579, 'name': 'Nightshades Premier Circ...",PTCGL,True,True,True,1,SWISS,5,BO1
1,69725552b1294bfab720364d,STANDARD,Turtwig Den Revival Series Challenge 41,2026-07-24,6,"{'id': 196, 'name': 'DabbingUkranian'}",PTCGL,True,True,True,1,SWISS,4,BO1
2,6a5baf52937230b102d3eb53,STANDARD,Team Corna Weekly July #5,2026-07-24,111,"{'id': 2583, 'name': 'Team Corna'}",PTCGL,True,True,True,1,SWISS,7,BO1
3,6a5baf52937230b102d3eb53,STANDARD,Team Corna Weekly July #5,2026-07-24,111,"{'id': 2583, 'name': 'Team Corna'}",PTCGL,True,True,True,2,SINGLE_ELIMINATION,4,BO1
4,6a621b37937230b102d423ab,STANDARD,Casual Friday!,2026-07-24,22,"{'id': 1216, 'name': 'Shiny Eevee Tourneys'}",PTCGL,True,True,True,1,SWISS,5,BO1


In [6]:
pairings = []
pairings_rate_limited = False

# One request per tournament, flattening every match into its own row
for tournament_id in df['id']:
    print(f"Fetching pairings for: {tournament_id}")
    pairings_url = f"https://play.limitlesstcg.com/api/tournaments/{tournament_id}/pairings"

    try:
        response = requests.get(pairings_url, timeout=TIMEOUT)
        response.raise_for_status()
        for match in response.json():
            pairings.append({"tournamentId": tournament_id, **match})
    except Exception as e:
        if is_rate_limited(e):
            print(f"429 Client Error on {tournament_id} - stopping iteration")
            pairings_rate_limited = True
            break
        print(f"Failed to fetch {tournament_id}: {e}")

    # Small sleep to respect the API rate limits
    time.sleep(TIMEOUT/1000)

# player2 is absent on byes, so make sure the column exists either way
pairings_df = pd.DataFrame(pairings).reindex(
    columns=["tournamentId", "phase", "round", "table", "player1", "player2", "winner"]
)
pairings_df["isBye"] = pairings_df["player2"].isna()

export_df(pairings_df, "pairings")
pairings_df.head()

Fetching pairings for: 69d519dd0e548b5c2fbe594c
Fetching pairings for: 69725552b1294bfab720364d
Fetching pairings for: 6a5baf52937230b102d3eb53
Fetching pairings for: 6a621b37937230b102d423ab
Fetching pairings for: 6a635f49937230b102d42f70
Fetching pairings for: 6a5925f1cc68177ec6c56c99
Fetching pairings for: 6a5f729b52c24ac2da6480ec
Fetching pairings for: 6a62967c52c24ac2da64a0c5
Fetching pairings for: 6a2acf672d97f3b0c2613d1c
Fetching pairings for: 6a60e6e3937230b102d41821
Fetching pairings for: 6a62800b937230b102d428c0
Fetching pairings for: 6a5e4d3752c24ac2da647443
Fetching pairings for: 6a60c513937230b102d4178c
Fetching pairings for: 6a50031c65724db9ded4178e
Fetching pairings for: 6a5e2d63937230b102d401a2
Fetching pairings for: 6a5e38dc937230b102d401fa
Fetching pairings for: 6a5a671b52c24ac2da644850
Fetching pairings for: 6a619256937230b102d4216b
Fetching pairings for: 6a613ff2937230b102d41ded
Fetching pairings for: 6a58f606cc68177ec6c56a6c
Fetching pairings for: 6a5e86f652c24ac2d

,tournamentId,phase,round,table,player1,player2,winner,isBye
0,69d519dd0e548b5c2fbe594c,1,1,NaN,kakalimagg,NaN,-1,True
1,69d519dd0e548b5c2fbe594c,1,1,NaN,ashc2017,NaN,-1,True
2,69d519dd0e548b5c2fbe594c,1,1,1.0,bdganeme,rafalelly123,rafalelly123,False
3,69d519dd0e548b5c2fbe594c,1,1,2.0,barnyt93,exaltedtcg,exaltedtcg,False
4,69d519dd0e548b5c2fbe594c,1,1,3.0,scottlm04,groshin,groshin,False


In [7]:
standings = []
standings_rate_limited = False

# One request per tournament, one row per player with deck/record flattened out
for tournament_id in df['id']:
    print(f"Fetching standings for: {tournament_id}")
    standings_url = f"https://play.limitlesstcg.com/api/tournaments/{tournament_id}/standings"

    try:
        response = requests.get(standings_url, timeout=TIMEOUT)
        response.raise_for_status()
        for entry in response.json():
            deck = entry.get("deck") or {}
            record = entry.get("record") or {}
            standings.append({
                "tournamentId": tournament_id,
                "placing": entry.get("placing"),
                "player": entry.get("player"),
                "name": entry.get("name"),
                "country": entry.get("country"),
                "deckId": deck.get("id"),
                "deckName": deck.get("name"),
                "deckIcons": deck.get("icons"),
                "wins": record.get("wins"),
                "losses": record.get("losses"),
                "ties": record.get("ties"),
                "drop": entry.get("drop"),
                "decklist": entry.get("decklist"),
            })
    except Exception as e:
        if is_rate_limited(e):
            print(f"429 Client Error on {tournament_id} - stopping iteration")
            standings_rate_limited = True
            break
        print(f"Failed to fetch {tournament_id}: {e}")

    # Small sleep to respect the API rate limits
    time.sleep(TIMEOUT/1000)

standings_df = pd.DataFrame(standings)

export_df(standings_df, "standings")
standings_df.head()

Fetching standings for: 69d519dd0e548b5c2fbe594c
Fetching standings for: 69725552b1294bfab720364d
429 Client Error on 69725552b1294bfab720364d - stopping iteration
Exported standings (32 rows) -> exports\standings_2026-07-24_183004.csv | exports\standings_2026-07-24_183004.xlsx


,tournamentId,placing,player,name,country,deckId,deckName,deckIcons,wins,losses,ties,drop,decklist
0,69d519dd0e548b5c2fbe594c,NaN,barnyt93,BarnyT93,DE,slowking-scr,Slowking,[slowking],0,3,0,3.0,"{'pokemon': [{'count': 4, 'set': 'SCR', 'numbe..."
1,69d519dd0e548b5c2fbe594c,NaN,peterpanda,PeterPanda,AR,beedrill-ex-cri,Beedrill,[beedrill],0,2,0,2.0,"{'pokemon': [{'count': 4, 'set': 'JTG', 'numbe..."
2,69d519dd0e548b5c2fbe594c,NaN,meneerwi,meneerwi,NL,lopunny-dudunsparce,Lopunny Dudunsparce,"[lopunny-mega, dudunsparce]",0,2,0,2.0,"{'pokemon': [{'count': 3, 'set': 'PBL', 'numbe..."
3,69d519dd0e548b5c2fbe594c,NaN,troopersp,troopersp,US,other,Other,[substitute],2,1,0,3.0,"{'pokemon': [{'count': 4, 'set': 'PBL', 'numbe..."
4,69d519dd0e548b5c2fbe594c,NaN,kakalimagg,MGC Kaue L,BR,dragapult-dusknoir,Dragapult Dusknoir,"[dragapult, dusknoir]",0,2,0,2.0,"{'pokemon': [{'count': 4, 'set': 'TWM', 'numbe..."


In [8]:
# One row per card in every decklist, tagged with its category (pokemon/trainer/energy)
cards = []

for row in standings_df.itertuples(index=False):
    decklist = row.decklist if isinstance(row.decklist, dict) else {}
    for category, entries in decklist.items():
        for card in entries or []:
            cards.append({
                "tournamentId": row.tournamentId,
                "player": row.player,
                "placing": row.placing,
                "deckId": row.deckId,
                "category": category,
                "count": card.get("count"),
                "name": card.get("name"),
                "set": card.get("set"),
                "number": card.get("number"),
            })

decklist_cards_df = pd.DataFrame(cards)

export_df(decklist_cards_df, "decklist_cards_df")
decklist_cards_df.head()

Exported decklist_cards_df (849 rows) -> exports\decklist_cards_df_2026-07-24_183004.csv | exports\decklist_cards_df_2026-07-24_183004.xlsx


,tournamentId,player,placing,deckId,category,count,name,set,number
0,69d519dd0e548b5c2fbe594c,barnyt93,NaN,slowking-scr,pokemon,4,Slowpoke,SCR,57
1,69d519dd0e548b5c2fbe594c,barnyt93,NaN,slowking-scr,pokemon,3,Slowking,SCR,58
2,69d519dd0e548b5c2fbe594c,barnyt93,NaN,slowking-scr,pokemon,2,Kyurem,SFA,47
3,69d519dd0e548b5c2fbe594c,barnyt93,NaN,slowking-scr,pokemon,2,Latias ex,SSP,76
4,69d519dd0e548b5c2fbe594c,barnyt93,NaN,slowking-scr,pokemon,2,Smoochum,SSP,75
